In [2]:
from pathlib import Path
from datetime import datetime, timezone
import time

import pandas as pd
import requests


# 8 coins for the project
COINS = {
    "BTC": {"coingecko_id": "bitcoin", "binance_symbol": "BTCUSDT", "cryptocompare_symbol": "BTC"},
    "ETH": {"coingecko_id": "ethereum", "binance_symbol": "ETHUSDT", "cryptocompare_symbol": "ETH"},
    "BNB": {"coingecko_id": "binancecoin", "binance_symbol": "BNBUSDT", "cryptocompare_symbol": "BNB"},
    "SOL": {"coingecko_id": "solana", "binance_symbol": "SOLUSDT", "cryptocompare_symbol": "SOL"},
    "XRP": {"coingecko_id": "ripple", "binance_symbol": "XRPUSDT", "cryptocompare_symbol": "XRP"},
    "ADA": {"coingecko_id": "cardano", "binance_symbol": "ADAUSDT", "cryptocompare_symbol": "ADA"},
    "DOGE": {"coingecko_id": "dogecoin", "binance_symbol": "DOGEUSDT", "cryptocompare_symbol": "DOGE"},
    "LTC": {"coingecko_id": "litecoin", "binance_symbol": "LTCUSDT", "cryptocompare_symbol": "LTC"},
}

START_DATE = "2021-01-01"
END_DATE = datetime.now(timezone.utc).strftime("%Y-%m-%d")

output_dir = Path("data")
output_dir.mkdir(parents=True, exist_ok=True)

session = requests.Session()
session.headers.update({"User-Agent": "crypto-ordinal-network-research/1.0"})


def to_unix_ms(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp() * 1000)


def to_unix_s(date_str: str) -> int:
    dt = datetime.strptime(date_str, "%Y-%m-%d").replace(tzinfo=timezone.utc)
    return int(dt.timestamp())


def safe_get(url: str, params: dict, retries: int = 4, sleep_s: float = 1.2):
    last_err = None
    for attempt in range(1, retries + 1):
        try:
            r = session.get(url, params=params, timeout=30)
            if r.status_code == 429:
                time.sleep(sleep_s * attempt)
                continue
            r.raise_for_status()
            return r.json()
        except Exception as exc:
            last_err = exc
            time.sleep(sleep_s * attempt)
    raise RuntimeError(f"Failed request after {retries} tries: {url} | params={params} | err={last_err}")


def fetch_binance_ohlcv(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://api.binance.com/api/v3/klines"
    start_ms = to_unix_ms(start_date)
    end_ms = to_unix_ms(end_date) + 86_400_000 - 1

    rows = []
    cur = start_ms

    while cur <= end_ms:
        params = {
            "symbol": symbol,
            "interval": "1d",
            "startTime": cur,
            "endTime": end_ms,
            "limit": 1000,
        }
        data = safe_get(url, params)
        if not data:
            break

        rows.extend(data)
        last_open_time = int(data[-1][0])
        next_cur = last_open_time + 86_400_000
        if next_cur <= cur:
            break
        cur = next_cur
        time.sleep(0.08)

    if not rows:
        return pd.DataFrame()

    df = pd.DataFrame(rows, columns=[
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_asset_volume",
        "number_of_trades",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore",
    ])

    df = df[["open_time", "open", "high", "low", "close", "volume", "number_of_trades"]].copy()
    df["date"] = pd.to_datetime(df["open_time"], unit="ms", utc=True).dt.date
    for c in ["open", "high", "low", "close", "volume", "number_of_trades"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df["source"] = "binance"
    return df[["date", "open", "high", "low", "close", "volume", "number_of_trades", "source"]]


def fetch_coingecko_daily(coin_id: str, start_date: str, end_date: str) -> pd.DataFrame:
    url = f"https://api.coingecko.com/api/v3/coins/{coin_id}/market_chart/range"
    params = {
        "vs_currency": "usd",
        "from": to_unix_s(start_date),
        "to": to_unix_s(end_date),
    }
    data = safe_get(url, params)

    prices = pd.DataFrame(data.get("prices", []), columns=["ts_ms", "close"])
    mcap = pd.DataFrame(data.get("market_caps", []), columns=["ts_ms", "market_cap"])
    tvol = pd.DataFrame(data.get("total_volumes", []), columns=["ts_ms", "total_volume"])

    if prices.empty:
        return pd.DataFrame()

    prices["date"] = pd.to_datetime(prices["ts_ms"], unit="ms", utc=True).dt.date
    mcap["date"] = pd.to_datetime(mcap["ts_ms"], unit="ms", utc=True).dt.date
    tvol["date"] = pd.to_datetime(tvol["ts_ms"], unit="ms", utc=True).dt.date

    # Keep last value per day in case API returns intraday points
    prices = prices.groupby("date", as_index=False)["close"].last()
    mcap = mcap.groupby("date", as_index=False)["market_cap"].last()
    tvol = tvol.groupby("date", as_index=False)["total_volume"].last()

    df = prices.merge(mcap, on="date", how="left").merge(tvol, on="date", how="left")
    df["open"] = pd.NA
    df["high"] = pd.NA
    df["low"] = pd.NA
    df["volume"] = df["total_volume"]
    df["number_of_trades"] = pd.NA
    df["source"] = "coingecko"
    return df[["date", "open", "high", "low", "close", "volume", "number_of_trades", "market_cap", "source"]]


def fetch_cryptocompare_histoday(symbol: str, start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://min-api.cryptocompare.com/data/v2/histoday"
    start_ts = to_unix_s(start_date)
    end_ts = to_unix_s(end_date)

    all_rows = []
    to_ts = end_ts

    while True:
        params = {
            "fsym": symbol,
            "tsym": "USD",
            "limit": 2000,
            "toTs": to_ts,
        }
        payload = safe_get(url, params)
        data = payload.get("Data", {}).get("Data", [])
        if not data:
            break

        all_rows.extend(data)
        earliest = min(int(x["time"]) for x in data)
        if earliest <= start_ts:
            break
        next_to_ts = earliest - 1
        if next_to_ts >= to_ts:
            break
        to_ts = next_to_ts
        time.sleep(0.08)

    if not all_rows:
        return pd.DataFrame()

    df = pd.DataFrame(all_rows)
    df = df[(df["time"] >= start_ts) & (df["time"] <= end_ts)].copy()
    if df.empty:
        return pd.DataFrame()

    df["date"] = pd.to_datetime(df["time"], unit="s", utc=True).dt.date
    keep_cols = ["date", "open", "high", "low", "close", "volumefrom", "source"]
    df["volume"] = pd.to_numeric(df["volumefrom"], errors="coerce")
    df["number_of_trades"] = pd.NA
    df["source"] = "cryptocompare"
    df = df[["date", "open", "high", "low", "close", "volume", "number_of_trades", "source"]]
    for c in ["open", "high", "low", "close", "volume"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


binance_frames = []
coingecko_frames = []
cryptocompare_frames = []

for ticker, cfg in COINS.items():
    print(f"Fetching {ticker}...")

    # Binance
    try:
        bdf = fetch_binance_ohlcv(cfg["binance_symbol"], START_DATE, END_DATE)
        if not bdf.empty:
            bdf["coin"] = ticker
            binance_frames.append(bdf)
            print(f"  Binance: {len(bdf)} rows")
        else:
            print("  Binance: no data")
    except Exception as exc:
        print(f"  Binance error: {exc}")

    # CoinGecko
    try:
        gdf = fetch_coingecko_daily(cfg["coingecko_id"], START_DATE, END_DATE)
        if not gdf.empty:
            gdf["coin"] = ticker
            coingecko_frames.append(gdf)
            print(f"  CoinGecko: {len(gdf)} rows")
        else:
            print("  CoinGecko: no data")
    except Exception as exc:
        print(f"  CoinGecko error: {exc}")

    # CryptoCompare
    try:
        cdf = fetch_cryptocompare_histoday(cfg["cryptocompare_symbol"], START_DATE, END_DATE)
        if not cdf.empty:
            cdf["coin"] = ticker
            cryptocompare_frames.append(cdf)
            print(f"  CryptoCompare: {len(cdf)} rows")
        else:
            print("  CryptoCompare: no data")
    except Exception as exc:
        print(f"  CryptoCompare error: {exc}")

    # Keep requests gentle to avoid rate limits
    time.sleep(0.35)


binance_df = pd.concat(binance_frames, ignore_index=True) if binance_frames else pd.DataFrame()
coingecko_df = pd.concat(coingecko_frames, ignore_index=True) if coingecko_frames else pd.DataFrame()
cryptocompare_df = pd.concat(cryptocompare_frames, ignore_index=True) if cryptocompare_frames else pd.DataFrame()

for df in [binance_df, coingecko_df, cryptocompare_df]:
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], utc=True)

if not binance_df.empty:
    binance_df.to_csv(output_dir / "binance_8coins_daily.csv", index=False)
if not coingecko_df.empty:
    coingecko_df.to_csv(output_dir / "coingecko_8coins_daily.csv", index=False)
if not cryptocompare_df.empty:
    cryptocompare_df.to_csv(output_dir / "cryptocompare_8coins_daily.csv", index=False)

combined = pd.concat(
    [df for df in [binance_df, coingecko_df, cryptocompare_df] if not df.empty],
    ignore_index=True,
) if any([not binance_df.empty, not coingecko_df.empty, not cryptocompare_df.empty]) else pd.DataFrame()

if not combined.empty:
    combined = combined.sort_values(["coin", "source", "date"]).reset_index(drop=True)
    combined.to_csv(output_dir / "crypto_8coins_daily_all_sources.csv", index=False)

print("\nDone.")
print(f"Date range: {START_DATE} to {END_DATE}")
print(f"Binance rows: {0 if binance_df.empty else len(binance_df):,}")
print(f"CoinGecko rows: {0 if coingecko_df.empty else len(coingecko_df):,}")
print(f"CryptoCompare rows: {0 if cryptocompare_df.empty else len(cryptocompare_df):,}")
print(f"Combined rows: {0 if combined.empty else len(combined):,}")
print(f"Saved to: {output_dir.resolve()}")

combined.head(10) if not combined.empty else "No combined data fetched."

Fetching BTC...
  Binance: 1945 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range | params={'vs_currency': 'usd', 'from': 1609459200, 'to': 1777420800} | err=401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range?vs_currency=usd&from=1609459200&to=1777420800
  CryptoCompare: 1945 rows
Fetching ETH...
  Binance: 1945 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/ethereum/market_chart/range | params={'vs_currency': 'usd', 'from': 1609459200, 'to': 1777420800} | err=None
  CryptoCompare: 1945 rows
Fetching BNB...
  Binance: 1945 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/binancecoin/market_chart/range | params={'vs_currency': 'usd', 'from': 1609459200, 'to': 1777420800} | err=None
  CryptoCompare: 1945 rows
Fetching SOL...
  Binance: 1945 rows
  CoinGecko error: Failed reque

,date,open,high,low,close,volume,number_of_trades,source,coin
0,2021-01-01 00:00:00+00:00,0.18134,0.18473,0.17000,0.17509,4.622016e+08,178824,binance,ADA
1,2021-01-02 00:00:00+00:00,0.17505,0.18457,0.16793,0.17742,6.603033e+08,231432,binance,ADA
2,2021-01-03 00:00:00+00:00,0.17740,0.20960,0.17255,0.20615,1.201451e+09,403926,binance,ADA
3,2021-01-04 00:00:00+00:00,0.20625,0.23992,0.19203,0.22528,1.463416e+09,548636,binance,ADA
4,2021-01-05 00:00:00+00:00,0.22518,0.26429,0.20696,0.25873,1.478888e+09,614385,binance,ADA
5,2021-01-06 00:00:00+00:00,0.25889,0.34772,0.25347,0.33309,2.059110e+09,893919,binance,ADA
6,2021-01-07 00:00:00+00:00,0.33311,0.35530,0.27850,0.29928,1.411322e+09,654373,binance,ADA
7,2021-01-08 00:00:00+00:00,0.29917,0.32165,0.26175,0.30389,1.284205e+09,531917,binance,ADA
8,2021-01-09 00:00:00+00:00,0.30392,0.33851,0.29583,0.33062,8.202675e+08,364494,binance,ADA
9,2021-01-10 00:00:00+00:00,0.33062,0.34120,0.27948,0.30153,9.086676e+08,409748,binance,ADA


In [3]:
# Fetch extended dataset (~10 years back from today) and overwrite CSV files
from datetime import timedelta

YEARS_BACK = 10
start_10y = (datetime.now(timezone.utc) - timedelta(days=365 * YEARS_BACK + 3)).strftime("%Y-%m-%d")
end_10y = datetime.now(timezone.utc).strftime("%Y-%m-%d")

print(f"Running extended fetch from {start_10y} to {end_10y}...")

binance_frames = []
coingecko_frames = []
cryptocompare_frames = []

for ticker, cfg in COINS.items():
    print(f"Fetching {ticker}...")

    try:
        bdf = fetch_binance_ohlcv(cfg["binance_symbol"], start_10y, end_10y)
        if not bdf.empty:
            bdf["coin"] = ticker
            binance_frames.append(bdf)
            print(f"  Binance: {len(bdf)} rows")
        else:
            print("  Binance: no data")
    except Exception as exc:
        print(f"  Binance error: {exc}")

    try:
        gdf = fetch_coingecko_daily(cfg["coingecko_id"], start_10y, end_10y)
        if not gdf.empty:
            gdf["coin"] = ticker
            coingecko_frames.append(gdf)
            print(f"  CoinGecko: {len(gdf)} rows")
        else:
            print("  CoinGecko: no data")
    except Exception as exc:
        print(f"  CoinGecko error: {exc}")

    try:
        cdf = fetch_cryptocompare_histoday(cfg["cryptocompare_symbol"], start_10y, end_10y)
        if not cdf.empty:
            cdf["coin"] = ticker
            cryptocompare_frames.append(cdf)
            print(f"  CryptoCompare: {len(cdf)} rows")
        else:
            print("  CryptoCompare: no data")
    except Exception as exc:
        print(f"  CryptoCompare error: {exc}")

    time.sleep(0.35)

binance_df_10y = pd.concat(binance_frames, ignore_index=True) if binance_frames else pd.DataFrame()
coingecko_df_10y = pd.concat(coingecko_frames, ignore_index=True) if coingecko_frames else pd.DataFrame()
cryptocompare_df_10y = pd.concat(cryptocompare_frames, ignore_index=True) if cryptocompare_frames else pd.DataFrame()

for df in [binance_df_10y, coingecko_df_10y, cryptocompare_df_10y]:
    if not df.empty:
        df["date"] = pd.to_datetime(df["date"], utc=True)

if not binance_df_10y.empty:
    binance_df_10y.to_csv(output_dir / "binance_8coins_daily.csv", index=False)
if not coingecko_df_10y.empty:
    coingecko_df_10y.to_csv(output_dir / "coingecko_8coins_daily.csv", index=False)
if not cryptocompare_df_10y.empty:
    cryptocompare_df_10y.to_csv(output_dir / "cryptocompare_8coins_daily.csv", index=False)

combined_10y = pd.concat(
    [df for df in [binance_df_10y, coingecko_df_10y, cryptocompare_df_10y] if not df.empty],
    ignore_index=True,
) if any([not binance_df_10y.empty, not coingecko_df_10y.empty, not cryptocompare_df_10y.empty]) else pd.DataFrame()

if not combined_10y.empty:
    combined_10y = combined_10y.sort_values(["coin", "source", "date"]).reset_index(drop=True)
    combined_10y.to_csv(output_dir / "crypto_8coins_daily_all_sources.csv", index=False)

print("\nExtended fetch done.")
print(f"Binance rows: {0 if binance_df_10y.empty else len(binance_df_10y):,}")
print(f"CoinGecko rows: {0 if coingecko_df_10y.empty else len(coingecko_df_10y):,}")
print(f"CryptoCompare rows: {0 if cryptocompare_df_10y.empty else len(cryptocompare_df_10y):,}")
print(f"Combined rows: {0 if combined_10y.empty else len(combined_10y):,}")
print(f"Saved to: {output_dir.resolve()}")

combined_10y.groupby(["source", "coin"]).size().reset_index(name="rows") if not combined_10y.empty else "No data fetched."

Running extended fetch from 2016-04-28 to 2026-04-29...
Fetching BTC...
  Binance: 3178 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range | params={'vs_currency': 'usd', 'from': 1461801600, 'to': 1777420800} | err=401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/bitcoin/market_chart/range?vs_currency=usd&from=1461801600&to=1777420800
  CryptoCompare: 3654 rows
Fetching ETH...
  Binance: 3178 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/ethereum/market_chart/range | params={'vs_currency': 'usd', 'from': 1461801600, 'to': 1777420800} | err=401 Client Error: Unauthorized for url: https://api.coingecko.com/api/v3/coins/ethereum/market_chart/range?vs_currency=usd&from=1461801600&to=1777420800
  CryptoCompare: 3654 rows
Fetching BNB...
  Binance: 3097 rows
  CoinGecko error: Failed request after 4 tries: https://api.coingecko.com/api/v3/coins/bin

,source,coin,rows
0,binance,ADA,2935
1,binance,BNB,3097
2,binance,BTC,3178
3,binance,DOGE,2491
4,binance,ETH,3178
5,binance,LTC,3060
6,binance,SOL,2088
7,binance,XRP,2918
8,cryptocompare,ADA,3654
9,cryptocompare,BNB,3654


In [7]:
# Build balanced datasets for fair cross-coin comparison (same date set for all 8 coins)
from pathlib import Path

import pandas as pd


data_dir = Path("data")
combined_path = data_dir / "crypto_8coins_daily_all_sources.csv"
if not combined_path.exists():
    raise FileNotFoundError(f"Missing file: {combined_path}")

all_df = pd.read_csv(combined_path, parse_dates=["date"])
required_coins = sorted(COINS.keys())
required_coin_count = len(required_coins)


def make_balanced(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    work = df.copy()
    work["date"] = pd.to_datetime(work["date"], utc=True)

    # Keep only dates where all required coins are present
    n_coins_per_date = work.groupby("date")["coin"].nunique()
    ok_dates = n_coins_per_date[n_coins_per_date == required_coin_count].index

    balanced = work[work["date"].isin(ok_dates) & work["coin"].isin(required_coins)].copy()
    balanced = balanced.sort_values(["source", "date", "coin"]).reset_index(drop=True)
    return balanced


# 1) Balanced by source (concatenated)
balanced_parts = []
for src in sorted(all_df["source"].dropna().unique()):
    src_df = all_df[all_df["source"] == src].copy()
    bal_src = make_balanced(src_df)
    if not bal_src.empty:
        balanced_parts.append(bal_src)

balanced_by_source = pd.concat(balanced_parts, ignore_index=True) if balanced_parts else pd.DataFrame()
if not balanced_by_source.empty:
    balanced_by_source.to_csv(data_dir / "crypto_8coins_daily_balanced_by_source.csv", index=False)

# 2) Balanced CryptoCompare only (usually longest common history)
cc_df = all_df[all_df["source"] == "cryptocompare"].copy()
cc_balanced = make_balanced(cc_df)
if not cc_balanced.empty:
    cc_balanced.to_csv(data_dir / "cryptocompare_8coins_daily_balanced.csv", index=False)

# 3) Wide close-price matrix from balanced CryptoCompare for ordinal-network workflows
close_wide = pd.DataFrame()
if not cc_balanced.empty:
    close_wide = (
        cc_balanced.pivot(index="date", columns="coin", values="close")
        .sort_index()
        .dropna(axis=0, how="any")
    )
    close_wide.to_csv(data_dir / "cryptocompare_8coins_close_wide_balanced.csv", index=True)

# 4) Clean wide matrix: keep days where every coin has strictly positive close
close_wide_positive = pd.DataFrame()
if not close_wide.empty:
    close_wide_positive = close_wide[(close_wide > 0).all(axis=1)].copy()
    close_wide_positive.to_csv(
        data_dir / "cryptocompare_8coins_close_wide_balanced_positive.csv", index=True
    )

print("Balanced datasets created.")
print(f"Required coins: {required_coins}")
print(f"Rows (balanced_by_source): {0 if balanced_by_source.empty else len(balanced_by_source):,}")
print(f"Rows (cc_balanced): {0 if cc_balanced.empty else len(cc_balanced):,}")
print(f"Rows (close_wide days): {0 if close_wide.empty else len(close_wide):,}")
print(
    f"Rows (close_wide_positive days): "
    f"{0 if close_wide_positive.empty else len(close_wide_positive):,}"
)

if not cc_balanced.empty:
    print(
        f"CryptoCompare balanced range: "
        f"{cc_balanced['date'].min()} -> {cc_balanced['date'].max()}"
    )

if not close_wide_positive.empty:
    print(
        f"Positive-only range: "
        f"{close_wide_positive.index.min()} -> {close_wide_positive.index.max()}"
    )

if not balanced_by_source.empty:
    display(
        balanced_by_source.groupby(["source", "coin"]).size().reset_index(name="rows").head(20)
    )

close_wide_positive.head(5) if not close_wide_positive.empty else "No positive-only balanced matrix created."

Balanced datasets created.
Required coins: ['ADA', 'BNB', 'BTC', 'DOGE', 'ETH', 'LTC', 'SOL', 'XRP']
Rows (balanced_by_source): 45,936
Rows (cc_balanced): 29,232
Rows (close_wide days): 3,654
Rows (close_wide_positive days): 2,211
CryptoCompare balanced range: 2016-04-28 00:00:00+00:00 -> 2026-04-29 00:00:00+00:00
Positive-only range: 2020-04-10 00:00:00+00:00 -> 2026-04-29 00:00:00+00:00


,source,coin,rows
0,binance,ADA,2088
1,binance,BNB,2088
2,binance,BTC,2088
3,binance,DOGE,2088
4,binance,ETH,2088
5,binance,LTC,2088
6,binance,SOL,2088
7,binance,XRP,2088
8,cryptocompare,ADA,3654
9,cryptocompare,BNB,3654


coin,ADA,BNB,BTC,DOGE,ETH,LTC,SOL,XRP
date,,,,,,,,
2020-04-10 00:00:00+00:00,0.03316,13.74,6876.49,0.002000,158.16,42.35,0.9496,0.1878
2020-04-11 00:00:00+00:00,0.03353,13.85,6887.60,0.001976,158.55,42.59,0.7973,0.1890
2020-04-12 00:00:00+00:00,0.03354,14.30,6913.76,0.002033,158.81,42.05,0.8830,0.1897
2020-04-13 00:00:00+00:00,0.03320,15.06,6859.43,0.001962,156.81,41.32,0.7772,0.1882
2020-04-14 00:00:00+00:00,0.03313,15.63,6879.44,0.002010,158.64,41.27,0.6664,0.1862
